<a href="https://colab.research.google.com/github/nishithkumar11/TradingSystemHealthCheck/blob/main/TradingSystem_HealthCheck_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
# ===============================================================
# LAYER 1 — MOUNT DRIVE + LOAD DATA (Realized P&L + Symbol)
# ===============================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
import matplotlib.pyplot as plt
import seaborn as sns

# Update this path to your actual file location in Drive
file_2026 = "/content/drive/MyDrive/Finance and Trading/Projects/HealthCheckComparison/pnl_2026.csv"

df_2026 = pd.read_csv(file_2026)

# Rename columns to standard names
df_2026 = df_2026.rename(columns={
    "Symbol": "symbol",
    "Realized P&L": "pnl"
})

print("Raw 2026 data (after renaming):")
display(df_2026.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raw 2026 data (after renaming):


,symbol,pnl
0,ACMESOLAR,514.20
1,ACUTAAS,853.65
2,AEROFLEX,16658.60
3,ANANDRATHI,2298.80
4,ANGELONE,-2762.75


In [31]:
# ===============================================================
# LAYER 2 — CLEANING
# ===============================================================

required_cols = ["symbol", "pnl"]
df_2026 = df_2026[required_cols].copy()

df_2026["pnl"] = pd.to_numeric(df_2026["pnl"], errors="coerce")
df_2026 = df_2026.dropna(subset=["pnl"])

print("Cleaned 2026 data:")
display(df_2026.head())

Cleaned 2026 data:


,symbol,pnl
0,ACMESOLAR,514.20
1,ACUTAAS,853.65
2,AEROFLEX,16658.60
3,ANANDRATHI,2298.80
4,ANGELONE,-2762.75


In [32]:
# ===============================================================
# LAYER 3 — METRIC FUNCTIONS
# ===============================================================

def win_rate(df):
    return (df['pnl'] > 0).mean()

def loss_rate(df):
    return (df['pnl'] < 0).mean()

def win_loss_ratio(df):
    wins = (df['pnl'] > 0).sum()
    losses = (df['pnl'] < 0).sum()
    return wins / max(1, losses)

def payoff_ratio(df):
    wins = df[df['pnl'] > 0]['pnl']
    losses = df[df['pnl'] < 0]['pnl']
    if len(wins) == 0 or len(losses) == 0:
        return np.nan
    return wins.mean() / abs(losses.mean())

def profit_factor(df):
    wins = df[df['pnl'] > 0]['pnl'].sum()
    losses = abs(df[df['pnl'] < 0]['pnl'].sum())
    if losses == 0:
        return np.nan
    return wins / losses

def expectancy_ratio(df):
    wr = win_rate(df)
    lr = loss_rate(df)
    wins = df[df['pnl'] > 0]['pnl']
    losses = df[df['pnl'] < 0]['pnl']
    if len(losses) == 0:
        return np.nan
    expectancy = (wr * wins.mean()) + (lr * losses.mean())
    return expectancy / abs(losses.mean())

def skewness_metric(df):
    return skew(df['pnl'])

def kurtosis_metric(df):
    return kurtosis(df['pnl'])

def tail_gain_ratio(df):
    wins = df[df['pnl'] > 0]['pnl']
    if len(wins) < 2:
        return np.nan
    return np.percentile(wins, 90) / np.percentile(wins, 50)

def tail_loss_ratio(df):
    losses = abs(df[df['pnl'] < 0]['pnl'])
    if len(losses) < 2:
        return np.nan
    return np.percentile(losses, 90) / np.percentile(losses, 50)

def coeff_variation(df):
    mean = df['pnl'].mean()
    std = df['pnl'].std()
    if mean == 0:
        return np.nan
    return std / mean

def iqr_std_ratio(df):
    iqr = df['pnl'].quantile(0.75) - df['pnl'].quantile(0.25)
    std = df['pnl'].std()
    if std == 0:
        return np.nan
    return iqr / std

def bootstrap_winrate(df, samples=3000):
    boot = []
    n = len(df)
    for _ in range(samples):
        s = df['pnl'].sample(n, replace=True)
        boot.append((s > 0).mean())
    return np.std(boot)

def compute_all(df):
    return {
        "win_rate": win_rate(df),
        "loss_rate": loss_rate(df),
        "win_loss_ratio": win_loss_ratio(df),
        "payoff_ratio": payoff_ratio(df),
        "profit_factor": profit_factor(df),
        "expectancy_ratio": expectancy_ratio(df),
        "skewness": skewness_metric(df),
        "kurtosis": kurtosis_metric(df),
        "tail_gain_ratio": tail_gain_ratio(df),
        "tail_loss_ratio": tail_loss_ratio(df),
        "coeff_variation": coeff_variation(df),
        "iqr_std_ratio": iqr_std_ratio(df),
        "bootstrap_winrate_std": bootstrap_winrate(df)
    }

In [33]:
# ===============================================================
# LAYER 4 — RUN METRICS
# ===============================================================

metrics_2026 = compute_all(df_2026)

metrics_2026_df = pd.DataFrame(metrics_2026, index=["2026"]).T
metrics_2026_df = metrics_2026_df.round(2)

print("\n================ METRICS FOR 2026 ================\n")
display(metrics_2026_df)


================ METRICS FOR 2026 ================



,2026
win_rate,0.39
loss_rate,0.59
win_loss_ratio,0.66
payoff_ratio,2.57
profit_factor,1.68
expectancy_ratio,0.41
skewness,3.94
kurtosis,19.48
tail_gain_ratio,3.48
tail_loss_ratio,2.75


In [34]:
# ===============================================================
# LAYER 7 — INTERPRETATION ENGINE (Improved Presentation)
# ===============================================================

# One-line human-friendly meanings for each metric
explanations = {
    "win_rate": "How often your trades make money.",
    "loss_rate": "How often your trades lose money.",
    "win_loss_ratio": "Whether you win more frequently than you lose.",
    "payoff_ratio": "How big your average win is compared to your average loss.",
    "profit_factor": "How much total money you make for every dollar you lose.",
    "expectancy_ratio": "How much you expect to make or lose per trade on average.",
    "skewness": "Whether your results lean toward big wins or big losses.",
    "kurtosis": "How extreme your outlier wins or losses are.",
    "tail_gain_ratio": "How strong your biggest winners are compared to normal winners.",
    "tail_loss_ratio": "How severe your biggest losses are compared to normal losses.",
    "coeff_variation": "How volatile or unstable your PnL is relative to its average.",
    "iqr_std_ratio": "How stable your PnL distribution is against extreme values.",
    "bootstrap_winrate_std": "How consistent your win rate is when the data is reshuffled."
}

# Ordered list: basic → intermediate → advanced
metric_order = [
    "win_rate",
    "loss_rate",
    "win_loss_ratio",
    "payoff_ratio",
    "profit_factor",
    "expectancy_ratio",
    "skewness",
    "kurtosis",
    "tail_gain_ratio",
    "tail_loss_ratio",
    "coeff_variation",
    "iqr_std_ratio",
    "bootstrap_winrate_std"
]

# Build interpretation table in correct order
rows_2026 = []
for metric in metric_order:
    rows_2026.append({
        "Metric": metric.replace("_", " ").title(),
        "Value (2026)": round(metrics_2026[metric], 2),
        "Meaning": explanations[metric]
    })

interpretation_2026 = pd.DataFrame(rows_2026)

# Apply clean left-aligned styling
styled_table = (
    interpretation_2026.style
    .set_properties(**{
        'text-align': 'left',
        'background-color': '#fafafa',
        'border-color': '#d0d0d0',
        'border-width': '1px',
        'border-style': 'solid',
        'padding': '6px'
    })
    .set_table_styles([
        {'selector': 'th', 'props': [
            ('background-color', '#2c3e50'),
            ('color', 'white'),
            ('font-weight', 'bold'),
            ('text-align', 'left'),
            ('padding', '8px')
        ]}
    ])
    .hide(axis="index")
)

print("\n================ 2026 METRIC MEANINGS ================\n")
styled_table


================ 2026 METRIC MEANINGS ================



Metric,Value (2026),Meaning
Win Rate,0.390000,How often your trades make money.
Loss Rate,0.590000,How often your trades lose money.
Win Loss Ratio,0.660000,Whether you win more frequently than you lose.
Payoff Ratio,2.570000,How big your average win is compared to your average loss.
Profit Factor,1.680000,How much total money you make for every dollar you lose.
Expectancy Ratio,0.410000,How much you expect to make or lose per trade on average.
Skewness,3.940000,Whether your results lean toward big wins or big losses.
Kurtosis,19.480000,How extreme your outlier wins or losses are.
Tail Gain Ratio,3.480000,How strong your biggest winners are compared to normal winners.
Tail Loss Ratio,2.750000,How severe your biggest losses are compared to normal losses.
